# Orivane prediction timestamp audit
2026-10-18. Existing cloud export, pinned main ledger, and read-only GitHub artifacts. No production API calls. 16 ET is a narrowly scoped screen; rows not flagged are not certified eligible. Source hashes are in manifest.json.


In [1]:
import json, csv
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter
root = Path.cwd()
rows = json.loads((root / "inputs.json").read_text())
ny = ZoneInfo("America/New_York")
# Audited interval only: June through October 2026. Not a production calendar.
holidays = {"2026-06-19", "2026-07-03", "2026-09-07"}
classified = []
for r in rows:
    if r["Ticker"] not in {"SPY", "QQQ", "NVDA", "^NDX", "^VIX"} or not r["Actual_1D_Date"]:
        continue
    assert "2026-06-01" <= r["As_Of_Date"] <= r["Actual_1D_Date"] < "2026-11-01"
    calendar_flags = [field for field in ["As_Of_Date", "Actual_1D_Date"] if datetime.fromisoformat(r[field]).weekday() >= 5 or r[field] in holidays]
    c = datetime.fromisoformat(r["Created_At"].replace("Z", "+00:00"))
    assert c.tzinfo is not None
    c = c.astimezone(ny)
    base = datetime.fromisoformat(r["As_Of_Date"] + "T16:00").replace(tzinfo=ny)
    opening = datetime.fromisoformat(r["Actual_1D_Date"] + "T09:30").replace(tzinfo=ny)
    close = datetime.fromisoformat(r["Actual_1D_Date"] + "T16:00").replace(tzinfo=ny)
    state = "base_before_16et_pending" if c < base else "late_after_outcome" if c >= close else "target_intraday_pending" if c >= opening else "timing_only_not_disqualified"
    # VIX before 16 ET is only a screen, not a claim about its official closing convention.
    classified.append({**r, "created_ny": c.isoformat(), "classification": state, "calendar_review_fields": ",".join(calendar_flags), "hours_after_outcome_close": round((c-close).total_seconds()/3600, 4)})
counts = {source: dict(Counter(r["classification"] for r in classified if r["source"] == source)) for source in ["cloud", "python"]}
assert counts["cloud"] == {"late_after_outcome": 22, "timing_only_not_disqualified": 110, "base_before_16et_pending": 17}
assert counts["python"] == {"timing_only_not_disqualified": 320, "base_before_16et_pending": 4, "late_after_outcome": 24}
flagged = [r for r in classified if r["classification"] != "timing_only_not_disqualified" or r["calendar_review_fields"]]
with (root / "flagged-samples.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(flagged[0]));writer.writeheader();writer.writerows(flagged)
print(json.dumps(counts, indent=2))
print("Calendar review required:", [(r["source"], r["Ticker"], r["As_Of_Date"], r["Actual_1D_Date"]) for r in classified if r["calendar_review_fields"]])
assert len(flagged) == 68
print("68 flagged rows: 46 late, 21 pre-base-close, 1 separate calendar review")
first = json.loads((root / "first-seen.json").read_text())
assert len(first) == 24 and all(r["timestamp_unchanged"] and r["forecast_unchanged"] for r in first)
assert all(not r["initial_actual_date"] for r in first)
print("24 Python rows: timestamps/forecasts unchanged since first Git appearance; initially unsettled")
artifacts = json.loads((root / "artifact-tail.json").read_text())
for run_id, as_of in [("33823899221", "2026-09-02"), ("36208754559", "2026-09-24")]:
    batch = [r for r in artifacts if r["run_id"] == run_id]
    assert all(r["input_last_date"] == as_of for r in batch if r["asset"] in {"SPY", "QQQ"})
    print(run_id, [(r["asset"], r["input_last_date"]) for r in batch])


{
  "cloud": {
    "timing_only_not_disqualified": 110,
    "late_after_outcome": 22,
    "base_before_16et_pending": 17
  },
  "python": {
    "timing_only_not_disqualified": 320,
    "base_before_16et_pending": 4,
    "late_after_outcome": 24
  }
}
Calendar review required: [('python', '^VIX', '2026-09-04', '2026-09-07')]
68 flagged rows: 46 late, 21 pre-base-close, 1 separate calendar review
24 Python rows: timestamps/forecasts unchanged since first Git appearance; initially unsettled
33823899221 [('NDX', '2026-09-03'), ('QQQ', '2026-09-02'), ('SPY', '2026-09-02'), ('USDCNY_X', '2026-09-04'), ('VIX', '2026-09-03')]
36208754559 [('NDX', '2026-09-25'), ('QQQ', '2026-09-24'), ('SPY', '2026-09-24'), ('USDCNY_X', '2026-09-25'), ('VIX', '2026-09-25')]
